Load Data & Cek Masalah Kualitas Data

In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv('credit_risk_dataset.csv', sep=';')

df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [12]:
print(f"Total baris awal : {len(df):,} baris, {df.shape[1]} kolom")
print(f"Jumlah duplikat  : {df.duplicated().sum():,} baris")
print("\nJumlah Missing Values per kolom:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Total baris awal : 32,581 baris, 12 kolom
Jumlah duplikat  : 165 baris

Jumlah Missing Values per kolom:
person_emp_length     895
loan_int_rate        3116
dtype: int64


Data Cleaning & Validasi Aturan Bisnis (Business Rules)

In [13]:
# Hapus data duplikat
df_clean = df.drop_duplicates().copy()

# Hapus anomali logika bisnis
anomali = (
    (df_clean['person_age'] > 80) |
    (df_clean['person_emp_length'] > 60) |
    (df_clean['person_emp_length'] > (df_clean['person_age'] - 14))
)
print(f"Anomali logika yang dihapus: {anomali.sum()} baris")
df_clean = df_clean[~anomali].copy()

# Tangani Missing Values
df_clean['person_emp_length'] = df_clean['person_emp_length'].fillna(
    df_clean['person_emp_length'].median()
)

df_clean['loan_int_rate'] = df_clean['loan_int_rate'].fillna(
    df_clean.groupby('loan_grade')['loan_int_rate'].transform('median')
)

print(f"Total baris bersih : {len(df_clean):,}")
print(f"Sisa missing values: {df_clean.isnull().sum().sum()}")

Anomali logika yang dihapus: 9 baris
Total baris bersih : 32,407
Sisa missing values: 0


Feature Engineering & Export ke CSV Standar

In [14]:
# Segmentasi Kelompok Usia
df_clean['age_group'] = pd.cut(
    df_clean['person_age'],
    bins=[19, 25, 35, 50, 80],
    labels=['20-25 (Gen Z)', '26-35 (Young Adult)', '36-50 (Mid Career)', '51+ (Senior)']
)

# Segmentasi Tingkat Pendapatan
df_clean['income_tier'] = pd.cut(
    df_clean['person_income'],
    bins=[0, 40000, 80000, 150000, np.inf],
    labels=['Low (<$40K)', 'Lower-Middle ($40K-$80K)', 'Upper-Middle ($80K-$150K)', 'High (>$150K)']
)

# Kategori Beban Cicilan terhadap Pendapatan (Debt-to-Income)
df_clean['dti_category'] = pd.cut(
    df_clean['loan_percent_income'],
    bins=[-0.01, 0.15, 0.30, 1.0],
    labels=['Low Burden (<=15%)', 'Moderate Burden (16%-30%)', 'High Burden (>30%)']
)

# Label Status Pinjaman yang Mudah Dibaca
df_clean['loan_status_label'] = df_clean['loan_status'].map({
    0: 'Non-Default (Lancar)',
    1: 'Default (Gagal Bayar)'
})

# Simpan sebagai CSV berpemisah koma (standar)
df_clean.to_csv('credit_risk_cleaned.csv', index=False)
print("Berhasil disimpan")

Berhasil disimpan
